In [ ]:
import os
from google.colab import drive
import pandas as pd
import uuid
import json
import numpy as np

In [ ]:
drive.mount('/content/drive')

In [ ]:
feature_directory = "/content/drive/MyDrive/Colab Notebooks/MastersResearchProject/dataset_features/"

# Dataset Loading

In [ ]:
import numpy as np, json, pickle, pandas as pd

def load_cutoff(cutoff_percentage, feature_directory, forum=None):
    """forum=None loads all 9; forum='anxiety' loads just that one."""
    dataframe = pd.read_csv(feature_directory + f"cutoff_{cutoff_percentage}_targets.csv")
    if forum is not None:
        dataframe = dataframe[dataframe["forum"] == forum].reset_index(drop=True)

    vector_columns = [
        "context_emotion_mean", "context_sbert_mean",
        "context_emotion_mean_with_post", "context_sbert_mean_with_post",
        "post_emotion", "post_sbert",
        "target_emotion_mean", "target_emotion_mean_poster",
    ]
    for column in vector_columns:
        dataframe[column] = dataframe[column].apply(
            lambda value: np.array(json.loads(value), dtype=np.float32) if isinstance(value, str) else None
        )

    with open(feature_directory + f"cutoff_{cutoff_percentage}_sequences.pkl", "rb") as file_handle:
        full_store = pickle.load(file_handle)
    # keep only the UIDs present after the forum filter
    keep_uids = set(dataframe["UID"])
    sequence_store = {uid: full_store[uid] for uid in keep_uids if uid in full_store}
    return dataframe, sequence_store


def build_dataset(dataframe, sequence_store, split, use_sequences=False,
                  include_post_in_sequence=True, require_poster_target=True):
    rows = dataframe[dataframe["split"] == split]
    if require_poster_target:
        rows = rows[rows["target_emotion_mean_poster"].notna()]

    post_emotion, post_sbert = [], []
    context_emotion, context_sbert = [], []
    context_emotion_seq, context_sbert_seq = [], []
    target_emotion_all, target_emotion_poster, poster_mask = [], [], []

    emotion_dim = len(dataframe["target_emotion_mean"].iloc[0]) # 43

    for _, row in rows.iterrows():
        uid = row["UID"]
        post_emotion.append(row["post_emotion"])
        post_sbert.append(row["post_sbert"])

        if use_sequences:
            store = sequence_store[uid]
            emo_key   = "emotion_with_post" if include_post_in_sequence else "emotion"
            sbert_key = "sbert_with_post"   if include_post_in_sequence else "sbert"
            context_emotion_seq.append(store[emo_key])
            context_sbert_seq.append(store[sbert_key])
        else:
            context_emotion.append(row["context_emotion_mean"])
            context_sbert.append(row["context_sbert_mean"])

        target_emotion_all.append(row["target_emotion_mean"])

        poster_target = row["target_emotion_mean_poster"]
        if poster_target is None or (isinstance(poster_target, float) and np.isnan(poster_target)):
            target_emotion_poster.append(np.full(emotion_dim, np.nan, dtype=np.float32))
            poster_mask.append(False)
        else:
            target_emotion_poster.append(np.asarray(poster_target, dtype=np.float32))
            poster_mask.append(True)

    dataset = {
        "post_emotion":          np.stack(post_emotion),
        "post_sbert":            np.stack(post_sbert),
        "target_emotion_all":    np.stack(target_emotion_all),
        "target_emotion_poster": np.stack(target_emotion_poster),
        "poster_mask":           np.array(poster_mask),
    }
    if use_sequences:
        dataset["context_emotion_seq"] = context_emotion_seq
        dataset["context_sbert_seq"]   = context_sbert_seq
    else:
        dataset["context_emotion"] = np.stack(context_emotion)
        dataset["context_sbert"]   = np.stack(context_sbert)
    return dataset

# Metrics

In [ ]:
def skill_per_dim(y_true, y_pred, train_mean):
    """1 - MSE_model/MSE_meanbaseline, per dimension"""
    sse_model = np.sum((y_true - y_pred) ** 2, axis=0)
    sse_mean  = np.sum((y_true - train_mean) ** 2, axis=0)
    skill = 1.0 - sse_model / np.where(sse_mean == 0, np.nan, sse_mean)
    return skill

def evaluate(y_true, y_pred, train_mean, mask=None):
    """Returns dict of metrics. mask (bool, per-row)"""
    if mask is not None:
        y_true, y_pred = y_true[mask], y_pred[mask]
    per_dim = skill_per_dim(y_true, y_pred, train_mean)
    cos = np.sum(y_true * y_pred, axis=1) / (
        np.linalg.norm(y_true, axis=1) * np.linalg.norm(y_pred, axis=1) + 1e-12)
    return {
        "n":              len(y_true),
        "mse":            float(np.mean((y_true - y_pred) ** 2)),
        "skill_macro":    float(np.nanmean(per_dim)),                       # equal weight per emotion dim
        "skill_pooled":   float(1.0 - np.sum((y_true - y_pred) ** 2)        # single global ratio
                                     / np.sum((y_true - train_mean) ** 2)),
        "skill_per_dim":  per_dim,
        "cosine":         float(np.mean(cos)),
    }

def show(name, m):
    print(f"{name:32s} | n={m['n']:5d} | MSE={m['mse']:.5f} | "
          f"skill_macro={m['skill_macro']:+.4f} | skill_pooled={m['skill_pooled']:+.4f} | "
          f"cos={m['cosine']:.4f}")

# Functions

In [ ]:
def feature_blocks(dataset, feature_set):
    """Return list of (name, array) blocks. Post vectors always included per matching modality."""
    blocks = []
    if feature_set in ("emotion", "both"):
        blocks.append(("post_emotion",    dataset["post_emotion"]))
        blocks.append(("context_emotion", dataset["context_emotion"]))
    if feature_set in ("bert", "both"):
        blocks.append(("post_sbert",    dataset["post_sbert"]))
        blocks.append(("context_sbert", dataset["context_sbert"]))
    return blocks

def standardize_blocks_fit(train_blocks, fit_rows):
    """Per-block z-score stats from train fit_rows only. Returns dict (mu,sd)."""
    stats = {}
    for name, arr in train_blocks:
        sub = arr[fit_rows]
        stats[name] = (sub.mean(0), sub.std(0) + 1e-8)
    return stats

def apply_blocks(blocks, stats):
    """Standardize each block with its own stats, then concatenate."""
    return np.concatenate([(arr - stats[name][0]) / stats[name][1]
                           for name, arr in blocks], axis=1)

# Stage 0: Creating and checking baseline models


In [ ]:
CUTOFFS = [25, 50, 75]

def target_and_context(dataset, which="all"):
    y = dataset["target_emotion_all"] if which == "all" else dataset["target_emotion_poster"]
    ctx = dataset["context_emotion"]
    mask = dataset["poster_mask"] if which == "poster" else None
    return y, ctx, mask

for cutoff in CUTOFFS:
    dataframe, sequence_store = load_cutoff(cutoff, feature_directory)
    train = build_dataset(dataframe, sequence_store, "train", require_poster_target=False)
    test  = build_dataset(dataframe, sequence_store, "test",  require_poster_target=False)

    print(f"\n===== CUTOFF {cutoff}% =====")
    for which in ["all", "poster"]:
        y_tr, ctx_tr, m_tr = target_and_context(train, which)
        y_te, ctx_te, m_te = target_and_context(test,  which)

        # train mean as constant predictor
        if which == "poster":
            train_mean = np.nanmean(np.where(m_tr[:, None], y_tr, np.nan), axis=0)
        else:
            train_mean = y_tr.mean(axis=0)

        mean_pred = np.tile(train_mean, (len(y_te), 1)) # mean baseline (skill=0 by construction)
        pers_pred = ctx_te # persistence: predict context mean unchanged

        print(f"  --- target: emotion_{which} ---")
        show("mean baseline", evaluate(y_te, mean_pred, train_mean, m_te))
        show("persistence (ctx=target)", evaluate(y_te, pers_pred, train_mean, m_te))

        # correlation of context mean vs target, per dim, averaged
        if which == "all":
            corrs = [np.corrcoef(ctx_te[:, d], y_te[:, d])[0, 1] for d in range(y_te.shape[1])]
            print(f"      mean |corr(context,target)| over dims: {np.nanmean(np.abs(corrs)):.4f}")

#Stage 1: floor (ridge + mean, 50%, pooled, average input, all 3 feature sets)

In [ ]:
from sklearn.linear_model import RidgeCV
import numpy as np

CUTOFF = 50
ALPHAS = [0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0]

dataframe, sequence_store = load_cutoff(CUTOFF, feature_directory)
train = build_dataset(dataframe, sequence_store, "train", require_poster_target=False)
test  = build_dataset(dataframe, sequence_store, "test",  require_poster_target=False)

for which in ["all", "poster"]:
    y_tr = train["target_emotion_all"] if which == "all" else train["target_emotion_poster"]
    y_te = test["target_emotion_all"]  if which == "all" else test["target_emotion_poster"]
    m_tr = None if which == "all" else train["poster_mask"]
    m_te = None if which == "all" else test["poster_mask"]

    if which == "poster":
        train_mean = np.nanmean(np.where(m_tr[:, None], y_tr, np.nan), axis=0)
        fit_rows = m_tr
    else:
        train_mean = y_tr.mean(axis=0)
        fit_rows = np.ones(len(y_tr), dtype=bool)

    print(f"\n===== STAGE 1 (RidgeCV)  cutoff {CUTOFF}%  target: emotion_{which} =====")
    show("mean baseline", evaluate(y_te, np.tile(train_mean, (len(y_te), 1)), train_mean, m_te))

    for feature_set in ["emotion", "bert", "both"]:
        tr_blocks = feature_blocks(train, feature_set)
        te_blocks = feature_blocks(test,  feature_set)

        stats = standardize_blocks_fit(tr_blocks, fit_rows)      # per-block, train-only
        Xtr = apply_blocks(tr_blocks, stats)
        Xte = apply_blocks(te_blocks, stats)

        ridge = RidgeCV(alphas=ALPHAS)                            # picks alpha by internal CV
        ridge.fit(Xtr[fit_rows], np.nan_to_num(y_tr)[fit_rows])

        pred = ridge.predict(Xte)
        m = evaluate(y_te, pred, train_mean, m_te)
        show(f"ridge [{feature_set}] (a={ridge.alpha_:g})", m)

Skill per dimension

In [ ]:
from transformers import AutoConfig
config = AutoConfig.from_pretrained("borisn70/bert-43-multilabel-emotion-detection")
id2label = config.id2label            # {0: 'admiration', 1: 'amusement', ...}

# make sure your emotion vectors are ordered by this same index — they are if
# predict_emotions returns logits/probs in the model's native label order
per_dim = evaluate(y_te, pred, train_mean, m_te)["skill_per_dim"]
order = np.argsort(per_dim)[::-1]
print("most predictable emotions:")
for d in order[:10]:
    print(f"  {id2label[d]:20s} skill {per_dim[d]:+.3f}")
print("least predictable:")
for d in order[-10:]:
    print(f"  {id2label[d]:20s} skill {per_dim[d]:+.3f}")

In [ ]:
from sklearn.linear_model import RidgeCV

train = build_dataset(dataframe, sequence_store, "train", require_poster_target=False)
y_tr = train["target_emotion_all"]; y_te = test["target_emotion_all"]
train_mean = y_tr.mean(axis=0)

tr_blocks = feature_blocks(train, "emotion"); te_blocks = feature_blocks(test, "emotion")
stats = standardize_blocks_fit(tr_blocks, np.ones(len(y_tr), dtype=bool))
Xtr = apply_blocks(tr_blocks, stats); Xte = apply_blocks(te_blocks, stats)
pred = RidgeCV(alphas=[0.1,1,10,100,1000,10000,100000]).fit(Xtr, y_tr).predict(Xte)

per_dim_skill = skill_per_dim(y_te, pred, train_mean)
per_dim_var   = y_te.var(axis=0)
finite = np.isfinite(per_dim_skill) & np.isfinite(per_dim_var) & (per_dim_var > 0)
print("usable dims:", finite.sum(), "/ 43")

slope, intercept = np.polyfit(per_dim_var[finite], per_dim_skill[finite], 1)
residual = per_dim_skill - (slope * per_dim_var + intercept)

order = np.argsort(residual)[::-1]
print("\npredictable BEYOND variance:")
for d in order[:6]:
    print(f"  {id2label[d]:16s} skill {per_dim_skill[d]:+.3f}  var {per_dim_var[d]:.5f}  resid {residual[d]:+.3f}")
print("less than variance predicts:")
for d in order[::-1][:6]:
    print(f"  {id2label[d]:16s} skill {per_dim_skill[d]:+.3f}  var {per_dim_var[d]:.5f}  resid {residual[d]:+.3f}")
print(f"\nslope: {slope:.2f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

per_dim_skill = skill_per_dim(y_te, pred, train_mean)
per_dim_var   = y_te.var(axis=0)
slope, intercept = np.polyfit(per_dim_var, per_dim_skill, 1)
residual = per_dim_skill - (slope * per_dim_var + intercept)

fig, ax = plt.subplots(figsize=(9, 6))

above = residual >= 0
ax.scatter(per_dim_var[above],  per_dim_skill[above],  s=40, color="#1baf7a", zorder=3, label="above fit (persists)")
ax.scatter(per_dim_var[~above], per_dim_skill[~above], s=40, color="#eb6834", zorder=3, label="below fit (reactive)")

xs = np.linspace(per_dim_var.min(), per_dim_var.max(), 50)
ax.plot(xs, slope * xs + intercept, color="#888780", lw=1.5, zorder=2, label=f"fit (slope={slope:.1f})")

label_idx = list(np.argsort(residual)[::-1][:6]) + list(np.argsort(residual)[:6])
for d in label_idx:
    ax.annotate(id2label[d], (per_dim_var[d], per_dim_skill[d]),
                xytext=(4, 3), textcoords="offset points", fontsize=8,
                color="#0F6E56" if residual[d] >= 0 else "#993C1D")

ax.set_xlabel("per-dimension target variance across threads")
ax.set_ylabel("per-dimension skill score")
ax.set_title("emotion predictability vs variance (points above the line persist beyond variance)")
ax.legend(fontsize=9, loc="lower right")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(feature_directory + "skill_vs_variance.png", dpi=150)
plt.show()

# Stage 2: Feature removal with MLP

In [ ]:
import torch, torch.nn as nn, numpy as np
from sklearn.linear_model import RidgeCV

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CUTOFF = 50

dataframe, sequence_store = load_cutoff(CUTOFF, feature_directory)
train = build_dataset(dataframe, sequence_store, "train", require_poster_target=False)
test  = build_dataset(dataframe, sequence_store, "test",  require_poster_target=False)

class MLP(nn.Module):
    def __init__(self, in_dim, out_dim=43, hidden=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_dim))
    def forward(self, x): return self.net(x)

def train_mlp(Xtr, Ytr, Xte, val_frac=0.15, epochs=400, patience=25, lr=1e-3, wd=1e-4):
    n = len(Xtr); idx = np.random.RandomState(0).permutation(n)
    n_val = int(val_frac * n); va, tr = idx[:n_val], idx[n_val:]
    Xt = torch.tensor(Xtr[tr], dtype=torch.float32, device=device)
    Yt = torch.tensor(Ytr[tr], dtype=torch.float32, device=device)
    Xv = torch.tensor(Xtr[va], dtype=torch.float32, device=device)
    Yv = torch.tensor(Ytr[va], dtype=torch.float32, device=device)
    model = MLP(Xtr.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.MSELoss()
    best_val, best_state, wait = np.inf, None, 0
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        loss_fn(model(Xt), Yt).backward(); opt.step()
        model.eval()
        with torch.no_grad(): vl = loss_fn(model(Xv), Yv).item()
        if vl < best_val - 1e-6: best_val, best_state, wait = vl, {k: v.clone() for k,v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        return model(torch.tensor(Xte, dtype=torch.float32, device=device)).cpu().numpy()

for which in ["all", "poster"]:
    y_tr = train["target_emotion_all"] if which=="all" else train["target_emotion_poster"]
    y_te = test["target_emotion_all"]  if which=="all" else test["target_emotion_poster"]
    m_tr = None if which=="all" else train["poster_mask"]
    m_te = None if which=="all" else test["poster_mask"]
    if which=="poster":
        train_mean = np.nanmean(np.where(m_tr[:,None], y_tr, np.nan), axis=0); fit = m_tr
    else:
        train_mean = y_tr.mean(axis=0); fit = np.ones(len(y_tr), dtype=bool)

    print(f"\n===== STAGE 2  cutoff {CUTOFF}%  target: emotion_{which} =====")
    show("mean baseline", evaluate(y_te, np.tile(train_mean,(len(y_te),1)), train_mean, m_te))

    for fs in ["emotion","bert","both"]:
        trb, teb = feature_blocks(train, fs), feature_blocks(test, fs)
        stats = standardize_blocks_fit(trb, fit)
        Xtr, Xte = apply_blocks(trb, stats), apply_blocks(teb, stats)
        # ridge reference
        rp = RidgeCV(alphas=[0.1,1,10,100,1000,10000,100000]).fit(Xtr[fit], np.nan_to_num(y_tr)[fit]).predict(Xte)
        show(f"ridge [{fs}]", evaluate(y_te, rp, train_mean, m_te))
        # mlp
        mp = train_mlp(Xtr[fit], np.nan_to_num(y_tr)[fit], Xte)
        show(f"MLP   [{fs}]", evaluate(y_te, mp, train_mean, m_te))

# Stage 3: Model comparison, Ridge, MLP, LSTM-avg, Transformer-avg

In [ ]:
import torch, torch.nn as nn, numpy as np
from sklearn.linear_model import RidgeCV

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CUTOFF = 50

dataframe, sequence_store = load_cutoff(CUTOFF, feature_directory)
train = build_dataset(dataframe, sequence_store, "train", require_poster_target=False)
test  = build_dataset(dataframe, sequence_store, "test",  require_poster_target=False)

class MLP(nn.Module):
    def __init__(s, d, o=43, h=256, p=0.3):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(d,h), nn.ReLU(), nn.Dropout(p),
                              nn.Linear(h,h), nn.ReLU(), nn.Dropout(p), nn.Linear(h,o))
    def forward(s,x): return s.net(x)

class SeqLSTM(nn.Module):                       # consumes (B,T,F); here T=1
    def __init__(s, f, o=43, h=128, p=0.3):
        super().__init__()
        s.lstm = nn.LSTM(f, h, batch_first=True)
        s.head = nn.Sequential(nn.Dropout(p), nn.Linear(h, o))
    def forward(s, x):
        out,_ = s.lstm(x); return s.head(out[:, -1])

class SeqTransformer(nn.Module):                # consumes (B,T,F); here T=1
    def __init__(s, f, o=43, d=128, heads=4, layers=2, p=0.3):
        super().__init__()
        s.proj = nn.Linear(f, d)
        enc = nn.TransformerEncoderLayer(d, heads, d*2, p, batch_first=True)
        s.enc = nn.TransformerEncoder(enc, layers)
        s.head = nn.Sequential(nn.Dropout(p), nn.Linear(d, o))
    def forward(s, x):
        return s.head(s.enc(s.proj(x)).mean(dim=1))

def train_torch(model, Xtr, Ytr, Xte, seq=False, val_frac=0.15, epochs=400, patience=25, lr=1e-3, wd=1e-4):
    n = len(Xtr); idx = np.random.RandomState(0).permutation(n)
    nv = int(val_frac*n); va, tr = idx[:nv], idx[nv:]
    def T(a): return torch.tensor(a, dtype=torch.float32, device=device)
    reshape = (lambda a: a[:, None, :]) if seq else (lambda a: a)   # add T=1 axis for seq models
    Xt, Yt, Xv, Yv = T(reshape(Xtr[tr])), T(Ytr[tr]), T(reshape(Xtr[va])), T(Ytr[va])
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    lf = nn.MSELoss(); best, bs, wait = np.inf, None, 0
    for _ in range(epochs):
        model.train(); opt.zero_grad(); lf(model(Xt), Yt).backward(); opt.step()
        model.eval()
        with torch.no_grad(): vl = lf(model(Xv), Yv).item()
        if vl < best-1e-6: best, bs, wait = vl, {k:v.clone() for k,v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(bs); model.eval()
    with torch.no_grad(): return model(T(reshape(Xte))).cpu().numpy()

for which in ["all", "poster"]:
    y_tr = train["target_emotion_all"] if which=="all" else train["target_emotion_poster"]
    y_te = test["target_emotion_all"]  if which=="all" else test["target_emotion_poster"]
    m_te = None if which=="all" else test["poster_mask"]
    if which=="poster":
        m_tr = train["poster_mask"]; train_mean = np.nanmean(np.where(m_tr[:,None], y_tr, np.nan), axis=0); fit = m_tr
    else:
        train_mean = y_tr.mean(axis=0); fit = np.ones(len(y_tr), dtype=bool)

    trb, teb = feature_blocks(train, "emotion"), feature_blocks(test, "emotion")
    stats = standardize_blocks_fit(trb, fit)
    Xtr, Xte = apply_blocks(trb, stats)[fit], apply_blocks(teb, stats)
    Ytr = np.nan_to_num(y_tr)[fit]
    d = Xtr.shape[1]

    print(f"\n===== STAGE 3  cutoff {CUTOFF}%  emotion features  target: emotion_{which} =====")
    show("mean baseline", evaluate(y_te, np.tile(train_mean,(len(y_te),1)), train_mean, m_te))
    show("ridge", evaluate(y_te, RidgeCV(alphas=[0.1,1,10,100,1000,10000,100000]).fit(Xtr,Ytr).predict(Xte), train_mean, m_te))
    show("MLP", evaluate(y_te, train_torch(MLP(d), Xtr, Ytr, Xte), train_mean, m_te))
    show("LSTM", evaluate(y_te, train_torch(SeqLSTM(d), Xtr, Ytr, Xte, seq=True), train_mean, m_te))
    show("Transformer", evaluate(y_te, train_torch(SeqTransformer(d), Xtr, Ytr, Xte, seq=True), train_mean, m_te))

# Stage 4: Sequence vs. average

In [ ]:
import torch, torch.nn as nn, numpy as np
from sklearn.linear_model import RidgeCV
from torch.nn.utils.rnn import pad_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CUTOFF = 50

dataframe, sequence_store = load_cutoff(CUTOFF, feature_directory)

def build_seq_split(dataframe, sequence_store, split, which="all", key="emotion_with_post"):
    rows = dataframe[dataframe["split"] == split]
    seqs, targets, keep = [], [], []
    for _, r in rows.iterrows():
        tgt = r["target_emotion_mean"] if which=="all" else r["target_emotion_mean_poster"]
        if which=="poster" and (tgt is None or not np.all(np.isfinite(np.asarray(tgt, dtype=np.float32)))):
            continue
        s = sequence_store[r["UID"]][key]                     # (T, 43)
        if s.shape[0] == 0: continue
        seqs.append(torch.tensor(np.nan_to_num(s), dtype=torch.float32))
        targets.append(np.asarray(tgt, dtype=np.float32))
    return seqs, np.stack(targets)

class SeqLSTM(nn.Module):
    def __init__(s, f, o=43, h=128, p=0.3):
        super().__init__(); s.lstm = nn.LSTM(f, h, batch_first=True)
        s.head = nn.Sequential(nn.Dropout(p), nn.Linear(h, o))
    def forward(s, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hn, _) = s.lstm(packed); return s.head(hn[-1])

class SeqTransformer(nn.Module):
    def __init__(s, f, o=43, d=128, heads=4, layers=2, p=0.3):
        super().__init__(); s.proj = nn.Linear(f, d)
        enc = nn.TransformerEncoderLayer(d, heads, d*2, p, batch_first=True)
        s.enc = nn.TransformerEncoder(enc, layers); s.head = nn.Sequential(nn.Dropout(p), nn.Linear(d, o))
    def forward(s, x, mask):                                  # mask: True where padded
        h = s.enc(s.proj(x), src_key_padding_mask=mask)
        h = h.masked_fill(mask.unsqueeze(-1), 0.0).sum(1) / (~mask).sum(1, keepdim=True)  # masked mean
        return s.head(h)

def collate(seqs, targets, idx):
    batch = [seqs[i] for i in idx]
    lengths = torch.tensor([len(b) for b in batch])
    padded = pad_sequence(batch, batch_first=True)                    # (B, Tmax, 43)
    mask = torch.arange(padded.shape[1])[None, :] >= lengths[:, None] # True where padded
    y = torch.tensor(targets[idx], dtype=torch.float32)
    return padded.to(device), lengths, mask.to(device), y.to(device)

def train_seq(model, seqs, targets, seqs_te, is_lstm, val_frac=0.15, epochs=400, patience=25, lr=1e-3, wd=1e-4, bs=64):
    n = len(seqs); idx = np.random.RandomState(0).permutation(n)
    nv = int(val_frac*n); va, tr = idx[:nv], idx[nv:]
    model = model.to(device); opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    lf = nn.MSELoss(); best, bs_state, wait = np.inf, None, 0
    def run(m_idx):
        Xp, ln, mk, y = collate(seqs, targets, m_idx)
        return (model(Xp, ln) if is_lstm else model(Xp, mk)), y
    for _ in range(epochs):
        model.train()
        perm = np.random.permutation(tr)
        for i in range(0, len(perm), bs):
            opt.zero_grad(); pred, y = run(perm[i:i+bs]); lf(pred, y).backward(); opt.step()
        model.eval()
        with torch.no_grad(): vp, vy = run(va); vl = lf(vp, vy).item()
        if vl < best-1e-6: best, bs_state, wait = vl, {k:v.clone() for k,v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(bs_state); model.eval()
    with torch.no_grad():
        Xp, ln, mk, _ = collate(seqs_te, np.zeros((len(seqs_te), 43), dtype=np.float32), np.arange(len(seqs_te)))
        return (model(Xp, ln) if is_lstm else model(Xp, mk)).cpu().numpy()

for which in ["all", "poster"]:
    tr_seqs, y_tr = build_seq_split(dataframe, sequence_store, "train", which)
    te_seqs, y_te = build_seq_split(dataframe, sequence_store, "test",  which)
    train_mean = y_tr.mean(axis=0)
    f = tr_seqs[0].shape[1]

    print(f"\n===== STAGE 4  cutoff {CUTOFF}%  SEQUENCE input  target: emotion_{which} =====")
    show("mean baseline", evaluate(y_te, np.tile(train_mean,(len(y_te),1)), train_mean))

    # ridge on averaged sequence as the linear reference on THIS exact sample set
    Xtr_avg = np.stack([s.mean(0).numpy() for s in tr_seqs])
    Xte_avg = np.stack([s.mean(0).numpy() for s in te_seqs])
    mu, sd = Xtr_avg.mean(0), Xtr_avg.std(0)+1e-8
    rp = RidgeCV(alphas=[0.1,1,10,100,1000,10000]).fit((Xtr_avg-mu)/sd, y_tr).predict((Xte_avg-mu)/sd)
    show("ridge [avg ref]", evaluate(y_te, rp, train_mean))

    show("LSTM [sequence]", evaluate(y_te, train_seq(SeqLSTM(f), tr_seqs, y_tr, te_seqs, True), train_mean))
    show("Transformer [sequence]", evaluate(y_te, train_seq(SeqTransformer(f), tr_seqs, y_tr, te_seqs, False), train_mean))

# Stage 5: Per-forum training vs. pooled vs. pooled-train/per-forum-test

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import RidgeCV

CUTOFF = 50
ALPHAS = [0.1, 1, 10, 100, 1000, 10000, 100000]

dataframe, sequence_store = load_cutoff(CUTOFF, feature_directory)
train = build_dataset(dataframe, sequence_store, "train", require_poster_target=False)
test  = build_dataset(dataframe, sequence_store, "test",  require_poster_target=False)

train_forum = dataframe[dataframe["split"]=="train"].reset_index(drop=True)["forum"].values
test_forum  = dataframe[dataframe["split"]=="test"].reset_index(drop=True)["forum"].values
y_tr, y_te = train["target_emotion_all"], test["target_emotion_all"]

Xtr = np.concatenate([train["post_emotion"], train["context_emotion"]], axis=1)
Xte = np.concatenate([test["post_emotion"],  test["context_emotion"]],  axis=1)

forums = sorted(set(train_forum))

def fit_predict(Xa, Ya, Xb):
    mu, sd = Xa.mean(0), Xa.std(0) + 1e-8
    return RidgeCV(alphas=ALPHAS).fit((Xa-mu)/sd, Ya).predict((Xb-mu)/sd)

# train one model per source (each forum + pooled)
sources = forums + ["POOLED"]
models_pred = {}   # source -> prediction on the FULL test set
for src in sources:
    tr_m = np.ones(len(y_tr), bool) if src=="POOLED" else (train_forum==src)
    if tr_m.sum() < 20:
        models_pred[src] = None; continue
    models_pred[src] = fit_predict(Xtr[tr_m], y_tr[tr_m], Xte)

# build matrix: rows=source trained on, cols=forum tested on; skill vs that forum's own train mean
matrix = pd.DataFrame(index=sources, columns=forums, dtype=float)
for tgt in forums:
    te_m = test_forum==tgt
    fmean = y_tr[train_forum==tgt].mean(axis=0)      # forum-local baseline
    for src in sources:
        if models_pred[src] is None: continue
        matrix.loc[src, tgt] = evaluate(y_te[te_m], models_pred[src][te_m], fmean)["skill_macro"]

# counts
tr_counts = {f: int((train_forum==f).sum()) for f in forums}
te_counts = {f: int((test_forum==f).sum())  for f in forums}

pd.set_option("display.width", 200, "display.max_columns", 20)
print("SKILL MATRIX  (row = trained on, col = tested on)  cutoff 50%, ridge/emotion, target=all\n")
short = [f[:12] for f in forums]
disp = matrix.copy(); disp.columns = short
print(disp.round(3).to_string())
print("\ntest n per forum: ", {f[:12]: te_counts[f] for f in forums})
print("train n per forum:", {f[:12]: tr_counts[f] for f in forums})
print("\ndiagonal (train-on-self) mean:", round(np.nanmean([matrix.loc[f,f] for f in forums]), 3))
print("POOLED row mean:               ", round(matrix.loc["POOLED"].mean(), 3))
print("off-diagonal (cross-forum) mean:",
      round(np.nanmean([matrix.loc[s,t] for s in forums for t in forums if s!=t]), 3))

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import RidgeCV

CUTOFF = 50
ALPHAS = [0.1, 1, 10, 100, 1000, 10000, 100000]

dataframe, sequence_store = load_cutoff(CUTOFF, feature_directory)
train = build_dataset(dataframe, sequence_store, "train", require_poster_target=False)
test  = build_dataset(dataframe, sequence_store, "test",  require_poster_target=False)
train_forum = dataframe[dataframe["split"]=="train"].reset_index(drop=True)["forum"].values
test_forum  = dataframe[dataframe["split"]=="test"].reset_index(drop=True)["forum"].values
y_tr, y_te = train["target_emotion_all"], test["target_emotion_all"]
Xtr = np.concatenate([train["post_emotion"], train["context_emotion"]], axis=1)
Xte = np.concatenate([test["post_emotion"],  test["context_emotion"]],  axis=1)
forums = sorted(set(train_forum))

def fit_predict(Xa, Ya, Xb):
    mu, sd = Xa.mean(0), Xa.std(0)+1e-8
    return RidgeCV(alphas=ALPHAS).fit((Xa-mu)/sd, Ya).predict((Xb-mu)/sd)

sources = forums + ["POOLED"]
targets = forums + ["POOLED"]                       # pooled test column added
preds = {}
for src in sources:
    tr_m = np.ones(len(y_tr), bool) if src=="POOLED" else (train_forum==src)
    preds[src] = fit_predict(Xtr[tr_m], y_tr[tr_m], Xte) if tr_m.sum()>=20 else None

global_mean = y_tr.mean(axis=0)                     # baseline for the pooled column
matrix = pd.DataFrame(index=sources, columns=targets, dtype=float)
for tgt in targets:
    if tgt == "POOLED":
        te_m = np.ones(len(y_te), bool); fmean = global_mean
    else:
        te_m = test_forum==tgt; fmean = y_tr[train_forum==tgt].mean(axis=0)
    for src in sources:
        if preds[src] is not None:
            matrix.loc[src, tgt] = evaluate(y_te[te_m], preds[src][te_m], fmean)["skill_macro"]

te_counts = {f: int((test_forum==f).sum()) for f in forums}; te_counts["POOLED"] = len(y_te)
tr_counts = {f: int((train_forum==f).sum()) for f in forums}; tr_counts["POOLED"] = len(y_tr)

def short(f): return "POOLED" if f=="POOLED" else f.replace("-", " ")[:16]
col_labels = [f"{short(f)}\n(te={te_counts[f]})" for f in targets]
row_labels = [f"{short(f)} (tr={tr_counts[f]})" for f in sources]

fig, ax = plt.subplots(figsize=(12, 9))
vmax = np.nanmax(np.abs(matrix.values))
sns.heatmap(matrix.astype(float), annot=True, fmt="+.3f", cmap="RdBu",
            center=0, vmin=-vmax, vmax=vmax, linewidths=0.5, linecolor="white",
            cbar_kws={"label": "skill score vs own mean"},
            xticklabels=col_labels, yticklabels=row_labels,
            annot_kws={"size": 8}, ax=ax)
ax.axhline(len(forums), color="black", lw=2)          # separate POOLED row
ax.axvline(len(forums), color="black", lw=2)          # separate POOLED column
ax.set_xlabel("tested on"); ax.set_ylabel("trained on")
ax.set_title("cross-forum transfer of emotion prediction (ridge/emotion, 50% cutoff)")
plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(feature_directory + "transfer_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("POOLED/POOLED headline skill:", round(matrix.loc["POOLED","POOLED"], 3))

# Stage 6: Cutoff sweep

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import RidgeCV

ALPHAS = [0.1, 1, 10, 100, 1000, 10000, 100000]
CUTOFFS = [25, 50, 75]

def fit_predict(Xa, Ya, Xb):
    mu, sd = Xa.mean(0), Xa.std(0)+1e-8
    return RidgeCV(alphas=ALPHAS).fit((Xa-mu)/sd, Ya).predict((Xb-mu)/sd)

results = []
for cutoff in CUTOFFS:
    dataframe, sequence_store = load_cutoff(cutoff, feature_directory)
    train = build_dataset(dataframe, sequence_store, "train", require_poster_target=False)
    test  = build_dataset(dataframe, sequence_store, "test",  require_poster_target=False)

    Xtr = np.concatenate([train["post_emotion"], train["context_emotion"]], axis=1)
    Xte = np.concatenate([test["post_emotion"],  test["context_emotion"]],  axis=1)

    for which in ["all", "poster"]:
        y_tr = train["target_emotion_all"] if which=="all" else train["target_emotion_poster"]
        y_te = test["target_emotion_all"]  if which=="all" else test["target_emotion_poster"]
        m_te = None if which=="all" else test["poster_mask"]
        if which=="poster":
            m_tr = train["poster_mask"]
            train_mean = np.nanmean(np.where(m_tr[:,None], y_tr, np.nan), axis=0)
            fit = m_tr
        else:
            train_mean = y_tr.mean(axis=0); fit = np.ones(len(y_tr), bool)

        pred = fit_predict(Xtr[fit], np.nan_to_num(y_tr)[fit], Xte)
        m = evaluate(y_te, pred, train_mean, m_te)
        results.append({"cutoff": f"{cutoff}%", "target": which, "n": m["n"],
                        "skill_macro": m["skill_macro"], "skill_pooled": m["skill_pooled"],
                        "mse": m["mse"]})

df = pd.DataFrame(results)
print(df.pivot(index="cutoff", columns="target", values="skill_macro").round(4).to_string())
print("\nfull:")
print(df.round(4).to_string(index=False))

# Stage 7: Do you need only poster and no comments

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import RidgeCV

ALPHAS = [0.1, 1, 10, 100, 1000, 10000, 100000]
CUTOFF = 50

dataframe, sequence_store = load_cutoff(CUTOFF, feature_directory)
train = build_dataset(dataframe, sequence_store, "train", require_poster_target=False)
test  = build_dataset(dataframe, sequence_store, "test",  require_poster_target=False)

def fit_predict(Xa, Ya, Xb):
    mu, sd = Xa.mean(0), Xa.std(0)+1e-8
    return RidgeCV(alphas=ALPHAS).fit((Xa-mu)/sd, Ya).predict((Xb-mu)/sd)

conditions = {
    "post only":            lambda d: d["post_emotion"],
    "comments only":        lambda d: d["context_emotion"],
    "post + comments":      lambda d: np.concatenate([d["post_emotion"], d["context_emotion"]], axis=1),
}

for which in ["all", "poster"]:
    y_tr = train["target_emotion_all"] if which=="all" else train["target_emotion_poster"]
    y_te = test["target_emotion_all"]  if which=="all" else test["target_emotion_poster"]
    m_te = None if which=="all" else test["poster_mask"]
    if which=="poster":
        m_tr = train["poster_mask"]
        train_mean = np.nanmean(np.where(m_tr[:,None], y_tr, np.nan), axis=0); fit = m_tr
    else:
        train_mean = y_tr.mean(axis=0); fit = np.ones(len(y_tr), bool)

    print(f"\n===== FINAL  cutoff {CUTOFF}%  ridge/emotion  target: emotion_{which} =====")
    show("mean baseline", evaluate(y_te, np.tile(train_mean,(len(y_te),1)), train_mean, m_te))
    for name, fn in conditions.items():
        pred = fit_predict(fn(train)[fit], np.nan_to_num(y_tr)[fit], fn(test))
        show(name, evaluate(y_te, pred, train_mean, m_te))